In [6]:
import pandas as pd
import numpy as np

# Define paths and load data
csv_path = r"C:\\Users\\jonas\\Documents\\Studium - PhD Informatik\\Survey Paper Interaction\\IMWUT26 Submission_01\\Major Revision\\Expert Evaluation\\Soscisurvey Data\\data_EarXplore_2026-06-26_10-53.csv"
codebook_path = r"C:\\Users\\jonas\\Documents\\Studium - PhD Informatik\\Survey Paper Interaction\\IMWUT26 Submission_01\\Major Revision\\Expert Evaluation\\Soscisurvey Data\\codebook_EarXplore_2026-06-26_10-54.xlsx"

# Load data
df = pd.read_csv(csv_path, sep='\t', encoding='utf-16')
codebook_df = pd.read_excel(codebook_path)

# Data cleaning: exclude test rows and rows with NaN status
rows_before = len(df)
df_clean = df[(df['SD01_01'] != 'test') & (df['STATUS'].notna())].copy()
rows_after = len(df_clean)
rows_removed = rows_before - rows_after

# Identify reasons for removal
test_rows = df[df['SD01_01'] == 'test']
nan_status_rows = df[df['STATUS'].isna()]
both_criteria = df[(df['SD01_01'] == 'test') & (df['STATUS'].isna())]

df_clean['Participant_ID'] = pd.to_numeric(df_clean['SD01_01'], errors='coerce')

print(f"✓ Data loaded and cleaned:")
print(f"   Original rows: {rows_before}")
print(f"   Rows removed: {rows_removed}")
print(f"     - Test entries (SD01_01='test'): {len(test_rows)}")
print(f"     - Incomplete/NaN status: {len(nan_status_rows)}")
print(f"     - (Overlap - meeting both criteria): {len(both_criteria)}")
print(f"   Valid responses retained: {rows_after}")

✓ Data loaded and cleaned:
   Original rows: 12
   Rows removed: 3
     - Test entries (SD01_01='test'): 1
     - Incomplete/NaN status: 3
     - (Overlap - meeting both criteria): 1
   Valid responses retained: 9


In [7]:
# Prepare analysis dataset: select, rename, and clean columns
column_mapping = {
    'CASE': 'Interview_Number',
    'SD01_01': 'Participant_ID', 
    'SD03_01': 'Age',
    'SD07_01': 'Experience',
    'SD02': 'Gender',
    'SD04': 'Industry_Academia',
    'SD05': 'Academia_Seniority',
    'SD06': 'Highest_Education'
}

# Select and rename columns
df_analysis = df_clean[list(column_mapping.keys())].copy()
df_analysis = df_analysis.rename(columns=column_mapping)

# Clean Experience: convert '4 years' to 4
df_analysis['Experience'] = df_analysis['Experience'].apply(
    lambda x: float(str(x).replace('years', '').replace('year', '').strip()) 
    if isinstance(x, str) and x.strip() else x
)

print(f"✓ Analysis dataset prepared: {df_analysis.shape[0]} rows, {df_analysis.shape[1]} columns")

✓ Analysis dataset prepared: 9 rows, 8 columns


In [8]:
# Check submission status for requested participant IDs
requested_ids = [33, 12, 21, 45, 85, 68, 78, 54, 97]
submission_status = []

print("\n" + "="*80)
print("SUBMISSION STATUS CHECK")
print("="*80)

for id_num in requested_ids:
    id_data = df_clean[df_clean['Participant_ID'] == id_num]
    
    if len(id_data) == 0:
        status = "❌ NOT SUBMITTED"
    elif all(id_data['FINISHED'] == 1):
        status = "✓ COMPLETED"
    else:
        status = "⚠ INCOMPLETE"
    
    submission_status.append({'ID': id_num, 'Status': status})
    
    if len(id_data) > 0:
        print(f"ID {id_num}: {status} (submitted: {id_data['STARTED'].iloc[0]})")
    else:
        print(f"ID {id_num}: {status}")

print("\n" + "="*80)


SUBMISSION STATUS CHECK
ID 33: ✓ COMPLETED (submitted: 2026-06-03 09:54:18)
ID 12: ✓ COMPLETED (submitted: 2026-06-08 11:44:21)
ID 21: ✓ COMPLETED (submitted: 2026-06-13 12:11:18)
ID 45: ✓ COMPLETED (submitted: 2026-06-12 09:50:33)
ID 85: ✓ COMPLETED (submitted: 2026-06-25 22:59:37)
ID 68: ✓ COMPLETED (submitted: 2026-06-05 13:53:44)
ID 78: ✓ COMPLETED (submitted: 2026-06-18 10:17:51)
ID 54: ✓ COMPLETED (submitted: 2026-06-19 17:43:11)
ID 97: ✓ COMPLETED (submitted: 2026-06-08 12:38:40)



In [9]:
# Add response labels for categorical variables (from codebook)
df_labeled = df_analysis.copy()

# Define response mappings from codebook
mappings = {
    'Gender': {1: 'Male', 2: 'Female', 3: 'Other', 4: 'Prefer not to say'},
    'Industry_Academia': {1: 'Academia', 2: 'Industry', 3: 'Both'},
    'Academia_Seniority': {
        1: 'Student', 2: 'PhD Student', 3: 'Post-Doc', 
        4: 'Assistant Professor', 5: 'Full Professor', 7: 'Associate Professor'
    },
    'Highest_Education': {
        1: 'A-Levels', 2: "Bachelor's", 3: "Master's", 4: 'PhD'
    }
}

# Add labeled columns
df_labeled['Gender_Label'] = df_labeled['Gender'].map(mappings['Gender'])
df_labeled['Industry_Academia_Label'] = df_labeled['Industry_Academia'].map(mappings['Industry_Academia'])
df_labeled['Academia_Seniority_Label'] = df_labeled['Academia_Seniority'].map(mappings['Academia_Seniority'])
df_labeled['Highest_Education_Label'] = df_labeled['Highest_Education'].map(mappings['Highest_Education'])

print(f"✓ Response labels added to dataset")

✓ Response labels added to dataset


In [10]:
print("\n" + "="*100)
print("DESCRIPTIVE STATISTICS - FINAL ANALYSIS")
print("="*100)

# 1. Age
age_clean = pd.to_numeric(df_analysis['Age'], errors='coerce')
print("\n1. AGE (years)")
print(f"   Description: Participant age")
print(f"   Type: Numeric | N={age_clean.notna().sum()}")
print(f"   Mean: {age_clean.mean():.1f} | Median: {age_clean.median():.1f} | Std: {age_clean.std():.2f}")
print(f"   Range: {age_clean.min():.0f} - {age_clean.max():.0f}")

# 2. Experience
exp_clean = pd.to_numeric(df_analysis['Experience'], errors='coerce')
print("\n2. EXPERIENCE (years)")
print(f"   Description: Years of experience")
print(f"   Type: Numeric | N={exp_clean.notna().sum()}")
print(f"   Mean: {exp_clean.mean():.2f} | Median: {exp_clean.median():.2f} | Std: {exp_clean.std():.2f}")
print(f"   Range: {exp_clean.min():.1f} - {exp_clean.max():.1f}")

# 3. Gender
print("\n3. GENDER")
print(f"   Description: Gender identity")
print(f"   Type: Categorical | N={df_labeled['Gender_Label'].notna().sum()}")
gender_dist = df_labeled['Gender_Label'].value_counts()
for label, count in gender_dist.items():
    print(f"     {label}: {count} ({count/len(df_labeled)*100:.1f}%)")

# 4. Industry or Academia
print("\n4. AFFILIATION")
print(f"   Description: Industry or Academia")
print(f"   Type: Categorical | N={df_labeled['Industry_Academia_Label'].notna().sum()}")
industry_dist = df_labeled['Industry_Academia_Label'].value_counts()
for label, count in industry_dist.items():
    print(f"     {label}: {count} ({count/len(df_labeled)*100:.1f}%)")

# 5. Academia Seniority
print("\n5. ACADEMIC SENIORITY")
print(f"   Description: Level within academia")
print(f"   Type: Ordinal | N={df_labeled['Academia_Seniority_Label'].notna().sum()}")
seniority_order = ['Student', 'PhD Student', 'Post-Doc', 'Assistant Professor', 'Associate Professor', 'Full Professor']
seniority_dist = df_labeled['Academia_Seniority_Label'].value_counts()
for level in seniority_order:
    if level in seniority_dist.index:
        count = seniority_dist[level]
        print(f"     {level}: {count} ({count/len(df_labeled)*100:.1f}%)")

# 6. Highest Education
print("\n6. HIGHEST EDUCATION")
print(f"   Description: Highest level of education")
print(f"   Type: Ordinal | N={df_labeled['Highest_Education_Label'].notna().sum()}")
edu_order = ['A-Levels', "Bachelor's", "Master's", 'PhD']
edu_dist = df_labeled['Highest_Education_Label'].value_counts()
for level in edu_order:
    if level in edu_dist.index:
        count = edu_dist[level]
        print(f"     {level}: {count} ({count/len(df_labeled)*100:.1f}%)")

print("\n" + "="*100)
print(f"SUMMARY: N={len(df_labeled)} valid expert responses | Date range: {df_clean['STARTED'].min()[:10]} to {df_clean['STARTED'].max()[:10]}")
print("="*100 + "\n")


DESCRIPTIVE STATISTICS - FINAL ANALYSIS

1. AGE (years)
   Description: Participant age
   Type: Numeric | N=9
   Mean: 33.6 | Median: 32.0 | Std: 6.19
   Range: 27 - 47

2. EXPERIENCE (years)
   Description: Years of experience
   Type: Numeric | N=9
   Mean: 4.61 | Median: 4.00 | Std: 2.26
   Range: 1.5 - 10.0

3. GENDER
   Description: Gender identity
   Type: Categorical | N=9
     Male: 7 (77.8%)
     Female: 2 (22.2%)

4. AFFILIATION
   Description: Industry or Academia
   Type: Categorical | N=9
     Academia: 8 (88.9%)
     Both: 1 (11.1%)

5. ACADEMIC SENIORITY
   Description: Level within academia
   Type: Ordinal | N=9
     PhD Student: 4 (44.4%)
     Post-Doc: 2 (22.2%)
     Assistant Professor: 2 (22.2%)
     Full Professor: 1 (11.1%)

6. HIGHEST EDUCATION
   Description: Highest level of education
   Type: Ordinal | N=9
     Master's: 3 (33.3%)
     PhD: 6 (66.7%)

SUMMARY: N=9 valid expert responses | Date range: 2026-06-03 to 2026-06-25

